# H3: Joint Fitness Model Comparison

M4 (joint W with ω + κ) vs M1 (effort-only), M2 (threat-only), M3 (single-parameter).

**Note:** SVI results shown as exploratory benchmarks. Confirmatory uses MCMC WAIC + PSIS-LOO.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, os
import matplotlib.pyplot as plt
from pathlib import Path

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10,
                     'axes.spines.right': False, 'axes.spines.top': False})

REPO_ROOT = Path(os.getcwd())
for _ in range(5):
    if (REPO_ROOT / '.git').exists(): break
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)

mc = pd.read_csv('results/stats/model_comparison_5models/comparison_table.csv')
# Rename M5→M4 to match current H3 numbering (joint model is now M4)
mc['Model_new'] = mc['Model'].map({'M1':'M1','M2':'M2','M3':'M3','M4':'M_sep','M5':'M4'})
mc_main = mc[mc['Model_new'].isin(['M1','M2','M3','M4'])].copy()
mc_main['Model'] = mc_main['Model_new']
m4_bic = mc_main.loc[mc_main['Model']=='M4','BIC'].values[0]
mc_main['dBIC'] = mc_main['BIC'] - m4_bic

print('SVI Model Comparison (exploratory benchmarks):')
print(mc_main[['Model','Description','choice_acc','choice_r2','vigor_r2','BIC','dBIC']].to_string(index=False))

SVI Model Comparison (exploratory benchmarks):
Model                Description  choice_acc  choice_r2  vigor_r2          BIC        dBIC
   M1            Effort-only (λ)    0.712337   0.952883       NaN 18191.161007 2000.656342
   M2            Threat-only (ω)    0.429272        NaN  0.495735 18521.128531 2330.623866
   M3         Single-channel (θ)    0.594176   0.006481  0.487291 19255.992545 3065.487880
   M4 Full model (λ+ω, shared S)    0.794483   0.951937  0.495879 16190.504665    0.000000


## H3a: M4 vs M1 — Threat matters beyond effort

In [2]:
m1 = mc_main[mc_main['Model']=='M1'].iloc[0]
m4 = mc_main[mc_main['Model']=='M4'].iloc[0]
dbic = m1['BIC'] - m4['BIC']
passed = dbic > 0

print('H3a — M4 vs M1 (effort-only)')
print('=' * 50)
print(f'  M1: acc={m1["choice_acc"]:.3f}, no vigor model')
print(f'  M4: acc={m4["choice_acc"]:.3f}, vigor r²={m4["vigor_r2"]:.3f}')
print(f'  ΔBIC = {dbic:+.0f}')
print(f'  H3a: {"PASS ✓" if passed else "FAIL ✗"}')

H3a — M4 vs M1 (effort-only)
  M1: acc=0.712, no vigor model
  M4: acc=0.794, vigor r²=0.496
  ΔBIC = +2001
  H3a: PASS ✓


## H3b: M4 vs M2 — Individual effort differences matter

In [3]:
m2 = mc_main[mc_main['Model']=='M2'].iloc[0]
dbic = m2['BIC'] - m4['BIC']
passed = dbic > 0

print('H3b — M4 vs M2 (threat-only)')
print('=' * 50)
print(f'  M2: acc={m2["choice_acc"]:.3f}, vigor r²={m2["vigor_r2"]:.3f}')
print(f'  M4: acc={m4["choice_acc"]:.3f}, vigor r²={m4["vigor_r2"]:.3f}')
print(f'  ΔBIC = {dbic:+.0f}')
print(f'  H3b: {"PASS ✓" if passed else "FAIL ✗"}')

H3b — M4 vs M2 (threat-only)
  M2: acc=0.429, vigor r²=0.496
  M4: acc=0.794, vigor r²=0.496
  ΔBIC = +2331
  H3b: PASS ✓


## H3c: M4 vs M3 — Capture cost and effort cost are separable

In [4]:
m3 = mc_main[mc_main['Model']=='M3'].iloc[0]
dbic = m3['BIC'] - m4['BIC']
passed = dbic > 0

print('H3c — M4 vs M3 (single-parameter)')
print('=' * 50)
print(f'  M3: acc={m3["choice_acc"]:.3f}, vigor r²={m3["vigor_r2"]:.3f}')
print(f'  M4: acc={m4["choice_acc"]:.3f}, vigor r²={m4["vigor_r2"]:.3f}')
print(f'  ΔBIC = {dbic:+.0f}')
print(f'  H3c: {"PASS ✓" if passed else "FAIL ✗"}')

H3c — M4 vs M3 (single-parameter)
  M3: acc=0.594, vigor r²=0.487
  M4: acc=0.794, vigor r²=0.496
  ΔBIC = +3065
  H3c: PASS ✓


## Summary

In [5]:
print('H3 SUMMARY')
print('=' * 60)
print(f'{"Test":<8} {"Alt":<5} {"ΔBIC":>8} {"Alt acc":>8} {"M4 acc":>8} {"Pass":>6}')
print('-' * 45)
for alt, hname in [('M1','H3a'),('M2','H3b'),('M3','H3c')]:
    row = mc_main[mc_main['Model']==alt].iloc[0]
    d = row['BIC'] - m4_bic
    p = '✓' if d > 0 else '✗'
    print(f'{hname:<8} {alt:<5} {d:>+8.0f} {row["choice_acc"]:>8.3f} {m4["choice_acc"]:>8.3f} {p:>6}')
n_pass = sum(1 for alt in ['M1','M2','M3'] if mc_main[mc_main['Model']==alt].iloc[0]['BIC'] - m4_bic > 0)
print(f'\n{n_pass}/3 tests pass.')

H3 SUMMARY
Test     Alt       ΔBIC  Alt acc   M4 acc   Pass
---------------------------------------------
H3a      M1       +2001    0.712    0.794      ✓
H3b      M2       +2331    0.429    0.794      ✓
H3c      M3       +3065    0.594    0.794      ✓

3/3 tests pass.
